# Семинар: текстовое ранжирование, часть II

На прошлом семинаре мы собрали TF-IDF руками и выяснили, что для русского главное — лемматизация.
Сегодня переходим к вероятностным моделям и честным метрикам.

План:
1. Данные: MIRACL — вопросы к русской Википедии, у пассажей есть заголовок, а в разметке есть и **нерелевантные** документы.
2. Метрики: AP/MAP, DCG и nDCG своими руками. Что делать с неразмеченными документами.
3. BM25 руками: насыщение `k1`, нормализация длины `b`, подбор параметров. Сравнение с TF-IDF.
4. Сверка с библиотекой `bm25s` — только для проверки, в ДЗ2 такие библиотеки запрещены.
5. Зоны: заголовок и текст. Конкатенация, смесь скоров и BM25F.
6. Языковая модель для поиска: query likelihood со сглаживанием Дирихле и Елинека — Мерсера.
7. Близость слов: бонус за термины запроса, стоящие рядом.
8. whoosh с BM25F и весами полей.
9. Сводная таблица и проверка значимости, задания, что дальше.

Перед запуском положите рядом с ноутбуком `docs.tsv`, `queries.tsv`, `qrels.tsv`.

In [1]:
# !pip install -r requirements.txt

## 0. Данные

MIRACL (Zhang et al., TACL 2023) — многоязычный бенчмарк поиска по Википедии. Мы берём русскую часть:
1252 вопроса и подвыборку корпуса — все размеченные пассажи плюс ~8 тыс. случайных, чтобы было из чего выбирать.

Важное отличие от RuBQ из прошлого семинара: асессоры MIRACL оценивали выдачу нескольких систем (pooling, см. лекцию),
поэтому у запроса есть не только релевантные (1), но и **размеченные нерелевантные** пассажи (0).
Нерелевантные здесь «трудные»: их нашли настоящие поисковые системы, они похожи на запрос по словам.

In [3]:
import collections
import math
import os
import re
import time
import unicodedata
import warnings
from functools import lru_cache

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=SyntaxWarning)

DOCS_TSV, QUERIES_TSV, QRELS_TSV = "docs.tsv", "queries.tsv", "qrels.tsv"
for path in (DOCS_TSV, QUERIES_TSV, QRELS_TSV):
    assert os.path.exists(path), f"Положите {path} рядом с ноутбуком"

docs = {}  # docid -> (title, text)
with open(DOCS_TSV, encoding="utf-8") as f:
    for line in f:
        docid, title, text = line.rstrip("\n").split("\t")
        docs[docid] = (title, text)

queries, split = {}, {}
with open(QUERIES_TSV, encoding="utf-8") as f:
    for line in f:
        qid, part, text = line.rstrip("\n").split("\t")
        queries[qid], split[qid] = text, part

qrels = collections.defaultdict(dict)  # qid -> {docid: 0/1}
with open(QRELS_TSV, encoding="utf-8") as f:
    for line in f:
        qid, docid, rel = line.split()
        qrels[qid][docid] = int(rel)

dev = [q for q in queries if split[q] == "dev"]
test = [q for q in queries if split[q] == "test"]
print(f"{len(docs)} пассажей, {len(queries)} запросов (dev {len(dev)}, test {len(test)})")

20712 пассажей, 1252 запросов (dev 400, test 852)


In [4]:
for qid in ("0", "3"):
    print("запрос:", queries[qid])
    for docid, rel in list(qrels[qid].items())[:4]:
        title, text = docs[docid]
        print(f"   rel={rel}  [{docid}] {title}: {text[:110]}...")
    print()

запрос: Когда начался Кари́бский кризис?
   rel=1  [105156#0] Карибский кризис: Кари́бский кризис — исторический термин, определяющий чрезвычайно напряжённое политическое, дипломатическое и ...
   rel=1  [36716#23] Громыко, Андрей Андреевич: Политическое, дипломатическое и военное противостояние СССР и США осенью 1962 года, известное в истории как Ка...
   rel=1  [6949493#75] Президентство Джона Кеннеди: В октябре 1962 года произошел Карибский кризис, ставший пиком гонки вооружений, когда мир оказался на грани во...
   rel=0  [105156#4] Карибский кризис: Первоначально, после победы революции на Кубе в 1959 году, у её лидера Фиделя Кастро не было тесных отношений ...

запрос: Какое расстояние от Коло́мны до Москвы?
   rel=1  [14080#1] Коломна: Коломна находится в центре европейской части России, на просторах Москворецко-Окской равнины. Город расположен...
   rel=1  [3015852#11] Рязанское направление Московской железной дороги: Первый пассажирский поезд по маршруту Москва — Коломна прошё

In [5]:
n_rel = [sum(r for r in qrels[q].values()) for q in queries]
n_non = [sum(1 - r for r in qrels[q].values()) for q in queries]
lens = [len(text.split()) for _, text in docs.values()]
tlens = [len(title.split()) for title, _ in docs.values()]

print(f"релевантных на запрос: в среднем {np.mean(n_rel):.1f}, размеченных нерелевантных: {np.mean(n_non):.1f}")
print(f"длина пассажа: медиана {np.median(lens):.0f} слов, заголовка: в среднем {np.mean(tlens):.1f} слова")
print("документ с docid 'статья#номер': пассажи одной статьи делят заголовок, например:",
      [d for d in docs if d.startswith("105156#")][:5])

релевантных на запрос: в среднем 2.8, размеченных нерелевантных: 7.6
длина пассажа: медиана 66 слов, заголовка: в среднем 2.6 слова
документ с docid 'статья#номер': пассажи одной статьи делят заголовок, например: ['105156#0', '105156#4', '105156#42', '105156#79']


## 1. Метрики: MAP и nDCG

На прошлом семинаре были Recall@10 и MRR@10. Сегодня добавим две метрики из лекции:

- **AP** (average precision): усредняем Precision@k по позициям, где стоят релевантные документы, и делим на число релевантных. **MAP** — среднее AP по запросам.
- **DCG@K** = Σ rel_i / log2(i + 1) — релевантность документа, «дисконтированная» по позиции. **nDCG@K** = DCG@K / IDCG@K, где IDCG — DCG идеальной выдачи.

Сначала проверим на игрушечном примере с градуированной релевантностью: 2 — отлично, 1 — частично, 0 — мимо.

In [6]:
def dcg(gains, k=10):
    return sum(g / math.log2(i + 2) for i, g in enumerate(gains[:k]))  # позиция i с нуля -> log2(i + 2)


def ndcg(gains, all_gains, k=10):
    ideal = dcg(sorted(all_gains, reverse=True), k)
    return dcg(gains, k) / ideal if ideal > 0 else 0.0


ranking = [1, 2, 0, 0, 2]            # оценки документов в порядке нашей выдачи
print("DCG@5 =", round(dcg(ranking, 5), 3))
print("IDCG@5 =", round(dcg(sorted(ranking, reverse=True), 5), 3))
print("nDCG@5 =", round(ndcg(ranking, ranking, 5), 3))

from sklearn.metrics import ndcg_score

# sklearn хочет «правильные» оценки и скоры модели; скоры = убывающие числа в порядке нашей выдачи
print("sklearn ndcg_score:", round(ndcg_score([ranking], [[5, 4, 3, 2, 1]], k=5), 3))

DCG@5 = 3.036
IDCG@5 = 3.762
nDCG@5 = 0.807
sklearn ndcg_score: 0.807


Совпало. Обратите внимание: документ с оценкой 2 на пятом месте почти не помогает — его вклад поделен на log2(6) ≈ 2.6.

Теперь функция оценки для настоящих выдач. Ранкер возвращает топ-100 docid, считаем MAP (по топ-100), nDCG@10, MRR@10 и Recall@100.

In [ ]:
def evaluate(run, qids, k=10):
    """run: qid -> список docid в порядке выдачи (не короче 100)."""
    ap = nd = rr = rec = 0.0
    for q in qids:
        rel = qrels[q]
        n_relevant = sum(rel.values())
        ranked = run[q]
        hits, prec_sum = 0, 0.0
        for i, d in enumerate(ranked[:100]):
            if rel.get(d, 0) > 0:
                hits += 1
                prec_sum += hits / (i + 1)
        ap += prec_sum / n_relevant
        nd += ndcg([rel.get(d, 0) for d in ranked], list(rel.values()), k)
        rr += next((1 / (i + 1) for i, d in enumerate(ranked[:k]) if rel.get(d, 0) > 0), 0.0)
        rec += hits / n_relevant
    n = len(qids)
    return {"MAP": round(ap / n, 3), f"nDCG@{k}": round(nd / n, 3), f"MRR@{k}": round(rr / n, 3),
            "Recall@100": round(rec / n, 3)}


def judged_at(run, qids, k=10):
    """Какая доля документов в топ-k вообще размечена (неважно, 0 или 1)."""
    return np.mean([np.mean([d in qrels[q] for d in run[q][:k]]) for q in qids])

## 2. BM25 руками

### 2.1 Предобработка

Та же лемматизация, что в части I, вместе с чисткой **знаков ударения**. В MIRACL они стоят не только в документах, но и в запросах («Кари́бский»):
это отдельный символ Unicode (combining acute accent), и без чистки слово не совпадёт с «карибский» из документа.

Здесь легко ошибиться. Соблазнительно разложить текст через NFD и выбросить **все** диакритические знаки (категория Unicode `Mn`).
Но NFD раскладывает и «й» на «и» + кратку, так что «который» превращается в «которыи», и лемматизатор начинает ошибаться.
Выбрасываем только ударения и собираем буквы обратно через NFC.

In [8]:
import pymorphy3

morph = pymorphy3.MorphAnalyzer()
WORD_RE = re.compile(r"\w+")


@lru_cache(maxsize=None)
def lemma(word):
    return morph.parse(word)[0].normal_form


ACCENTS = {"\u0301", "\u0300"}  # знаки ударения


def normalize(text):
    # NFD отделяет знак ударения от буквы: «и́» -> «и» + U+0301. Выбрасываем только ударения,
    # а NFC собирает остальное обратно -- иначе вместе с ними пропала бы кратка в «й»
    text = unicodedata.normalize("NFD", text.lower())
    text = "".join(ch for ch in text if ch not in ACCENTS)
    return unicodedata.normalize("NFC", text).replace("ё", "е")


def normalize_naive(text):
    """Как делать НЕ надо: выбросить все диакритические знаки."""
    text = unicodedata.normalize("NFD", text.lower())
    return "".join(ch for ch in text if unicodedata.category(ch) != "Mn")


def preprocess(text):
    return [lemma(w) for w in WORD_RE.findall(normalize(text))]


q = queries["0"]
print(q)
print("без чистки:     ", [lemma(w) for w in WORD_RE.findall(q.lower())])
print("все знаки Mn:   ", [lemma(w) for w in WORD_RE.findall(normalize_naive(q))])
print("только ударения:", preprocess(q))

Когда начался Кари́бский кризис?
без чистки:      ['когда', 'начаться', 'карий', 'бский', 'кризис']
все знаки Mn:    ['когда', 'начаться', 'карибския', 'кризис']
только ударения: ['когда', 'начаться', 'карибский', 'кризис']


In [9]:
start = time.time()
doc_ids = sorted(docs)
title_toks = [preprocess(docs[d][0]) for d in doc_ids]
body_toks = [preprocess(docs[d][1]) for d in doc_ids]
Q = {q: preprocess(t) for q, t in queries.items()}
N = len(doc_ids)
print(f"лемматизировали {N} пассажей и {len(Q)} запросов за {time.time() - start:.0f} сек")

лемматизировали 20712 пассажей и 1252 запросов за 9 сек


### 2.2 Индекс и формула

Обратный индекс как в части I: термин → список (номер документа, tf). Плюс длины документов — они нужны BM25.
Пока ищем по склейке заголовка и текста.

Формула из лекции:

BM25(q, d) = Σ IDF(t) · tf · (k1 + 1) / (tf + k1 · (1 − b + b · |d| / avgdl))

IDF — «сглаженный» RSV-вариант из вывода BIM: log(1 + (N − df + 0.5) / (df + 0.5)). Единица под логарифмом
нужна, чтобы IDF не уходил в минус у слов, которые есть больше чем в половине документов.

In [ ]:
import heapq


def build_index(token_lists):
    postings = collections.defaultdict(list)
    lengths = np.zeros(len(token_lists))
    for i, toks in enumerate(token_lists):
        lengths[i] = len(toks)
        for term, tf in collections.Counter(toks).items():
            postings[term].append((i, tf))
    return dict(postings), lengths


full_toks = [t + b for t, b in zip(title_toks, body_toks)]
postings, doc_len = build_index(full_toks)
avgdl = doc_len.mean()
print(f"терминов: {len(postings)}, средняя длина документа: {avgdl:.1f}")


def idf_bm25(df):
    return math.log(1 + (N - df + 0.5) / (df + 0.5))


def bm25_scores(query_terms, k1=1.2, b=0.75):
    scores = collections.defaultdict(float)  # аккумуляторы, как в части I: term-at-a-time
    for term in set(query_terms):
        plist = postings.get(term)
        if not plist:
            continue
        idf = idf_bm25(len(plist))
        for i, tf in plist:
            scores[i] += idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * doc_len[i] / avgdl))
    return scores


def top_k(scores, k=100):
    return [doc_ids[i] for i, _ in heapq.nlargest(k, scores.items(), key=lambda kv: kv[1])] # сложность heapq.nlargest?? n * log(K)


q = "0"
print(queries[q])
for rank, d in enumerate(top_k(bm25_scores(Q[q]), 5), 1):
    print(f"  {rank}. rel={qrels[q].get(d, '?')} {docs[d][0]}: {docs[d][1][:90]}")

терминов: 94975, средняя длина документа: 82.5
Когда начался Кари́бский кризис?
  1. rel=1 Карибский кризис: Кари́бский кризис — исторический термин, определяющий чрезвычайно напряжённое политическое
  2. rel=? Кризис семи лет: Кризис семи лет ( — решение, поворотный пункт) — кризис в развитии ребёнка, который возник
  3. rel=0 Карибский кризис: В пятницу, 26 октября, в 13:00 по вашингтонскому времени в Белый дом поступило сообщение о
  4. rel=? На полях Фландрии: Первая мировая война — один из самых кровопролитных конфликтов в истории человечества. Она
  5. rel=? Финансовый кризис 2007—2008 годов: Начальной стадией мирового финансово-экономического кризиса послужил ипотечный кризис в СШ


`rel=?` — документ не размечен. Сравним с лучшим TF-IDF из части I (сублинейный tf `1 + log tf`, сглаженный IDF, косинус) на тех же леммах.

In [11]:
idf_sk = {t: 1 + math.log((N + 1) / (len(p) + 1)) for t, p in postings.items()}
norm = np.zeros(N)
for t, plist in postings.items():
    for i, tf in plist:
        norm[i] += ((1 + math.log(tf)) * idf_sk[t]) ** 2
norm = np.sqrt(norm)
norm[norm == 0] = 1.0


def tfidf_scores(query_terms):
    scores = collections.defaultdict(float)
    for t, qtf in collections.Counter(query_terms).items():
        for i, tf in postings.get(t, []):
            scores[i] += (1 + math.log(qtf)) * idf_sk[t] * (1 + math.log(tf)) * idf_sk[t] / norm[i]
    return scores


start = time.time()
runs = {"TF-IDF (часть I)": {q: top_k(tfidf_scores(Q[q])) for q in test}}
runs["BM25 (k1=1.2, b=0.75)"] = {q: top_k(bm25_scores(Q[q])) for q in test}
print(f"{time.time() - start:.1f} сек на {len(test)} запросов x 2 метода")
pd.DataFrame({name: evaluate(run, test) for name, run in runs.items()}).T

10.4 сек на 852 запросов x 2 метода


,MAP,nDCG@10,MRR@10,Recall@100
TF-IDF (часть I),0.531,0.626,0.622,0.968
"BM25 (k1=1.2, b=0.75)",0.569,0.661,0.659,0.969


In [12]:
print("judged@10:", {name: round(judged_at(run, test), 3) for name, run in runs.items()})

judged@10: {'TF-IDF (часть I)': np.float64(0.664), 'BM25 (k1=1.2, b=0.75)': np.float64(0.72)}


**judged@10** — доля размеченных документов в топ-10. Всё, что не размечено, `evaluate` считает нерелевантным.
Если новая модель начнёт находить хорошие, но неразмеченные документы, метрики её несправедливо накажут —
это та самая проблема pooling из лекции. Следите за judged@k, когда сравниваете очень разные модели.

### 2.3 Насыщение: что делает k1

Вклад термина как функция tf при документе средней длины (b не влияет). У TF-IDF он растёт линейно.

In [13]:
tfs = [1, 2, 3, 5, 10, 20]
table = {f"k1={k1}": [round(tf * (k1 + 1) / (tf + k1), 2) for tf in tfs] for k1 in (0.5, 1.2, 2.0, 5.0)}
table["TF-IDF (сырой tf)"] = tfs
pd.DataFrame(table, index=[f"tf={t}" for t in tfs]).T

,tf=1,tf=2,tf=3,tf=5,tf=10,tf=20
k1=0.5,1.0,1.20,1.29,1.36,1.43,1.46
k1=1.2,1.0,1.38,1.57,1.77,1.96,2.08
k1=2.0,1.0,1.50,1.80,2.14,2.50,2.73
k1=5.0,1.0,1.71,2.25,3.00,4.00,4.80
TF-IDF (сырой tf),1.0,2.00,3.00,5.00,10.00,20.00


При k1 = 1.2 двадцать вхождений дают всего вдвое больше одного: keyword stuffing из части I больше не работает.
Предел равен k1 + 1, поэтому k1 → 0 превращает BM25 в «есть слово или нет», а k1 → ∞ возвращает линейный tf.

### 2.4 Подбор k1 и b: векторизуем

Перебирать сетку словарной реализацией долго. Заметим, что вес пары (термин, документ) не зависит от запроса.
Значит, его можно **посчитать заранее** для всех ненулевых tf и положить в разреженную матрицу. Скор запроса — сумма
столбцов его терминов. Ровно так устроена библиотека `bm25s` и impact-индексы из лекции.

In [14]:
import scipy.sparse as sp

vocab = {t: j for j, t in enumerate(postings)}
rows, cols, vals = [], [], []
for t, plist in postings.items():
    for i, tf in plist:
        rows.append(i)
        cols.append(vocab[t])
        vals.append(tf)
TF = sp.coo_matrix((vals, (rows, cols)), shape=(N, len(vocab)), dtype=np.float32)
DF = np.bincount(TF.col, minlength=len(vocab))
IDF = np.log(1 + (N - DF + 0.5) / (DF + 0.5))


def bm25_matrix(k1=1.2, b=0.75, tf=TF, lengths=doc_len):
    w = tf.copy()
    norm_len = 1 - b + b * lengths[w.row] / lengths.mean()
    w.data = IDF[w.col] * w.data * (k1 + 1) / (w.data + k1 * norm_len)
    return w.tocsc()


def run_matrix(W, qids, k=100):
    run = {}
    for q in qids:
        cols_q = [vocab[t] for t in set(Q[q]) if t in vocab]
        s = np.asarray(W[:, cols_q].sum(axis=1)).ravel()
        idx = np.argpartition(-s, k)[:k]
        run[q] = [doc_ids[i] for i in idx[np.argsort(-s[idx])]]
    return run


start = time.time()
check = run_matrix(bm25_matrix(), test)
print(f"матричная версия: {time.time() - start:.2f} сек на {len(test)} запросов")
print("совпадает со словарной:", evaluate(check, test) == evaluate(runs["BM25 (k1=1.2, b=0.75)"], test))

матричная версия: 0.20 сек на 852 запросов
совпадает со словарной: True


In [15]:
grid = {}
for k1 in (0.3, 0.6, 0.9, 1.2, 1.5, 2.0, 3.0):
    grid[f"k1={k1}"] = {f"b={b}": evaluate(run_matrix(bm25_matrix(k1, b), dev), dev)["nDCG@10"]
                        for b in (0.0, 0.3, 0.5, 0.75, 1.0)}
grid = pd.DataFrame(grid).T
grid.round(3)

,b=0.0,b=0.3,b=0.5,b=0.75,b=1.0
k1=0.3,0.613,0.636,0.642,0.644,0.643
k1=0.6,0.621,0.646,0.651,0.653,0.645
k1=0.9,0.622,0.654,0.665,0.654,0.640
k1=1.2,0.613,0.651,0.659,0.649,0.632
k1=1.5,0.607,0.648,0.655,0.650,0.627
k1=2.0,0.599,0.644,0.646,0.643,0.614
k1=3.0,0.580,0.628,0.634,0.631,0.594


In [16]:
best_k1, best_b = max(((float(r[3:]), float(c[2:])) for r in grid.index for c in grid.columns),
                      key=lambda kb: grid.loc[f"k1={kb[0]}", f"b={kb[1]}"])
print(f"лучшее на dev: k1={best_k1}, b={best_b}")
runs[f"BM25 (k1={best_k1}, b={best_b})"] = run_matrix(bm25_matrix(best_k1, best_b), test)
pd.DataFrame({name: evaluate(run, test) for name, run in runs.items()}).T

лучшее на dev: k1=0.9, b=0.5


,MAP,nDCG@10,MRR@10,Recall@100
TF-IDF (часть I),0.531,0.626,0.622,0.968
"BM25 (k1=1.2, b=0.75)",0.569,0.661,0.659,0.969
"BM25 (k1=0.9, b=0.5)",0.568,0.660,0.653,0.970


BM25 уверенно обходит TF-IDF: nDCG@10 0.661 против 0.626, MAP 0.569 против 0.531. Это та же лемматизация и тот же индекс — выигрыш даёт только формула: насыщение tf и нормировка длины.

Подбор на dev дал k1 = 0.9, b = 0.5, но на test это ровно то же качество, что у стандартных k1 = 1.2, b = 0.75 (0.660 против 0.661). Поверхность в районе стандартных значений плоская, поэтому 1.2 / 0.75 — разумный выбор по умолчанию. Хуже всего на сетке — b = 0 (нормировки длины нет) и большие k1 (почти линейный tf).

## 3. Сверка с библиотекой bm25s

`bm25s` (Lù, 2024) — быстрая реализация на NumPy/SciPy с тем же предрасчётом весов.
Проверим, что наша формула совпадает с её вариантом `lucene`. **В ДЗ2 готовые ранжирующие библиотеки запрещены** —
здесь она нужна только как эталон для отладки.

In [17]:
import bm25s

start = time.time()
retriever = bm25s.BM25(k1=1.2, b=0.75, method="lucene")
retriever.index(full_toks, show_progress=False)
res, _ = retriever.retrieve([Q[q] for q in test], k=100, show_progress=False)
lib_run = {q: [doc_ids[i] for i in res[j]] for j, q in enumerate(test)}
print(f"bm25s: индекс + {len(test)} запросов за {time.time() - start:.1f} сек")
pd.DataFrame({"наш BM25": evaluate(runs["BM25 (k1=1.2, b=0.75)"], test), "bm25s": evaluate(lib_run, test)}).T

bm25s: индекс + 852 запросов за 0.8 сек


,MAP,nDCG@10,MRR@10,Recall@100
наш BM25,0.569,0.661,0.659,0.969
bm25s,0.568,0.660,0.657,0.969


Метрики совпадают до третьего знака. Мелкое расхождение — это ничьи по скору и float32 внутри библиотеки. Если ваш BM25 в ДЗ заметно расходится с `bm25s` на тех же токенах — ищите ошибку в формуле.

## 4. Зоны: заголовок и текст

У пассажа MIRACL есть заголовок — название статьи Википедии. Слово запроса в заголовке — сильный сигнал.
Три способа это учесть:

- **(a)** склеить заголовок и текст — то, что мы делали до сих пор;
- **(b)** посчитать BM25 по каждому полю отдельно и сложить с весом: w · BM25(title) + BM25(text);
- **(c)** BM25F: сначала сложить tf (tw) по полям с весами и своей нормировкой длины, а насыщение применить **один раз** к сумме.

Разница (b) и (c) в том, что в (b) насыщение применяется в каждом поле отдельно: слово, которое есть и в заголовке, и в тексте, получает почти двойной вклад.

In [18]:
t_post, t_len = build_index(title_toks)
b_post, b_len = build_index(body_toks)


def field_matrix(field_postings):
    r, c, v = [], [], []
    for t, plist in field_postings.items():
        for i, tf in plist:
            r.append(i)
            c.append(vocab[t])
            v.append(tf)
    return sp.coo_matrix((v, (r, c)), shape=(N, len(vocab)), dtype=np.float32)


TF_title, TF_body = field_matrix(t_post), field_matrix(b_post)

# (b) линейная смесь
W_title = bm25_matrix(best_k1, best_b, TF_title, t_len)
W_body = bm25_matrix(best_k1, best_b, TF_body, b_len)
mix = {w: evaluate(run_matrix(w * W_title + W_body, dev), dev)["nDCG@10"] for w in (0.5, 1, 2, 3)}
print("смесь, nDCG@10 на dev по весу заголовка:", mix)

смесь, nDCG@10 на dev по весу заголовка: {0.5: 0.642, 1: 0.65, 2: 0.639, 3: 0.626}


In [19]:
def bm25f_matrix(w_title=2.0, k1=1.2, b_title=0.5, b_body=0.75):
    """BM25F: tf полей нормируем по длине своего поля, взвешиваем, складываем, потом насыщаем."""
    def field_part(tf, lengths, b):
        m = tf.tocoo(copy=True)
        m.data = m.data / (1 - b + b * lengths[m.row] / lengths.mean())
        return m.tocsr()

    pseudo_tf = (w_title * field_part(TF_title, t_len, b_title) + field_part(TF_body, b_len, b_body)).tocoo()
    pseudo_tf.data = IDF[pseudo_tf.col] * pseudo_tf.data * (k1 + 1) / (pseudo_tf.data + k1)
    return pseudo_tf.tocsc()


bm25f_grid = {w: evaluate(run_matrix(bm25f_matrix(w, best_k1), dev), dev)["nDCG@10"] for w in (1, 2, 3, 5, 8)}
print("BM25F, nDCG@10 на dev по весу заголовка:", bm25f_grid)
best_w = max(bm25f_grid, key=bm25f_grid.get)
best_mix = max(mix, key=mix.get)

runs[f"смесь BM25(title)·{best_mix} + BM25(text)"] = run_matrix(best_mix * W_title + W_body, test)
runs[f"BM25F (w_title={best_w})"] = run_matrix(bm25f_matrix(best_w, best_k1), test)
pd.DataFrame({name: evaluate(run, test) for name, run in runs.items()}).T

BM25F, nDCG@10 на dev по весу заголовка: {1: 0.66, 2: 0.67, 3: 0.672, 5: 0.668, 8: 0.664}


,MAP,nDCG@10,MRR@10,Recall@100
TF-IDF (часть I),0.531,0.626,0.622,0.968
"BM25 (k1=1.2, b=0.75)",0.569,0.661,0.659,0.969
"BM25 (k1=0.9, b=0.5)",0.568,0.660,0.653,0.970
смесь BM25(title)·1 + BM25(text),0.549,0.643,0.636,0.974
BM25F (w_title=3),0.587,0.677,0.672,0.973


- Смесь (b) **хуже** простой склейки: 0.643 против 0.661 по nDCG@10, и лучший вес заголовка на dev — всего 1. Слово, которое есть и в заголовке, и в тексте, насыщается дважды и получает почти двойной вклад. У пассажей одной статьи общий заголовок, поэтому все они разом «выпрыгивают» вверх.
- BM25F (c) — лучший метод на сегодня: 0.677 по nDCG@10 при весе заголовка 3. Насыщение одно на сумму полей, поэтому совпадение в заголовке — сильный, но ограниченный сигнал.
- Вывод для ДЗ2: зоны в VK MARCO надо учитывать именно как в BM25F, а не складывать два BM25.

## 5. Языковая модель для поиска

Query likelihood: ранжируем документы по вероятности, что языковая модель документа «сгенерировала» запрос.
Модель документа — униграммная, как в лекции 3, только без контекста. И так же, как там, нужна **сглаживание**:
без него документ без одного из слов запроса получает вероятность 0.

- Дирихле: p(t | d) = (tf + μ · p(t | C)) / (|d| + μ)
- Елинек — Мерсер: p(t | d) = (1 − λ) · tf / |d| + λ · p(t | C)

где p(t | C) — частота термина во всей коллекции.

Считать log P(q | d) по всем документам честно — долго. Но если расписать формулу Дирихле, получится

log P(q | d) = Σ_{t ∈ q ∩ d} log(1 + tf / (μ · p(t | C)))  +  |q| · log(μ / (|d| + μ))  +  (слагаемое, не зависящее от d)

Первая сумма — только по совпавшим терминам (обратный индекс снова работает), и 1 / p(t | C) играет роль IDF.
Второе слагаемое — нормировка длины. Проверим, что разложение верное.

In [20]:
cf = np.asarray(TF.tocsc().sum(axis=0)).ravel()   # частота термина в коллекции
p_c = cf / cf.sum()


def ql_full(q_terms, i, mu):
    """Честный log P(q|d) по всем словам запроса."""
    tf_d = collections.Counter(full_toks[i])
    return sum(math.log((tf_d[t] + mu * p_c[vocab[t]]) / (doc_len[i] + mu)) for t in q_terms if t in vocab)


def ql_decomposed(q_terms, i, mu):
    tf_d = collections.Counter(full_toks[i])
    q_in = [t for t in q_terms if t in vocab]
    matched = sum(math.log(1 + tf_d[t] / (mu * p_c[vocab[t]])) for t in q_in if tf_d[t] > 0)
    constant = sum(math.log(p_c[vocab[t]]) for t in q_in)   # одинаково для всех документов
    return matched + len(q_in) * math.log(mu / (doc_len[i] + mu)) + constant


q, i = "0", doc_ids.index(runs["BM25 (k1=1.2, b=0.75)"]["0"][0])
print(round(ql_full(Q[q], i, 1000), 6), round(ql_decomposed(Q[q], i, 1000), 6))

-26.544878 -26.544878


In [21]:
def lm_dirichlet_run(qids, mu=1000, k=100):
    run = {}
    for q in qids:
        q_in = [t for t in Q[q] if t in vocab]
        scores = collections.defaultdict(float)
        for t in q_in:  # каждое вхождение термина в запрос считается отдельно
            for i, tf in postings[t]:
                scores[i] += math.log(1 + tf / (mu * p_c[vocab[t]]))
        final = {i: s + len(q_in) * math.log(mu / (doc_len[i] + mu)) for i, s in scores.items()}
        run[q] = top_k(final, k)
    return run


def lm_jm_run(qids, lam=0.5, k=100):
    run = {}
    for q in qids:
        scores = collections.defaultdict(float)
        for t in (t for t in Q[q] if t in vocab):
            for i, tf in postings[t]:
                scores[i] += math.log(1 + (1 - lam) * tf / (doc_len[i] * lam * p_c[vocab[t]]))
        run[q] = top_k(scores, k)
    return run


start = time.time()
lm_grid = {mu: evaluate(lm_dirichlet_run(dev, mu), dev)["nDCG@10"] for mu in (50, 200, 500, 1000, 2000)}
jm_grid = {lam: evaluate(lm_jm_run(dev, lam), dev)["nDCG@10"] for lam in (0.1, 0.3, 0.5, 0.7, 0.9)}
print(f"({time.time() - start:.0f} сек)")
print("Дирихле, nDCG@10 на dev по μ:", lm_grid)
print("Елинек — Мерсер, nDCG@10 на dev по λ:", jm_grid)
best_mu, best_lam = max(lm_grid, key=lm_grid.get), max(jm_grid, key=jm_grid.get)
runs[f"LM Дирихле (μ={best_mu})"] = lm_dirichlet_run(test, best_mu)
runs[f"LM Елинек — Мерсер (λ={best_lam})"] = lm_jm_run(test, best_lam)
pd.DataFrame({name: evaluate(run, test) for name, run in runs.items()}).T

(29 сек)
Дирихле, nDCG@10 на dev по μ: {50: 0.628, 200: 0.64, 500: 0.637, 1000: 0.639, 2000: 0.632}
Елинек — Мерсер, nDCG@10 на dev по λ: {0.1: 0.615, 0.3: 0.624, 0.5: 0.629, 0.7: 0.642, 0.9: 0.636}


,MAP,nDCG@10,MRR@10,Recall@100
TF-IDF (часть I),0.531,0.626,0.622,0.968
"BM25 (k1=1.2, b=0.75)",0.569,0.661,0.659,0.969
"BM25 (k1=0.9, b=0.5)",0.568,0.660,0.653,0.970
смесь BM25(title)·1 + BM25(text),0.549,0.643,0.636,0.974
BM25F (w_title=3),0.587,0.677,0.672,0.973
LM Дирихле (μ=200),0.558,0.649,0.641,0.968
LM Елинек — Мерсер (λ=0.7),0.566,0.654,0.651,0.966


Разложение верное: оба числа совпадают. Внутри query likelihood с Дирихле уже есть и TF, и IDF (через 1 / p(t | C)), и нормировка длины. TF-IDF получается из вероятностной модели, а не только из эвристики.

По качеству обе языковые модели чуть уступают BM25: 0.649 (Дирихле) и 0.654 (Елинек — Мерсер) против 0.661 по nDCG@10. Лучшее μ на dev — 200, а не классические 2000: у нас короткие пассажи (медиана 66 слов), и сглаживать их надо меньше. У Елинека — Мерсера лучшее λ = 0.7.

## 6. Близость слов

Мешок слов не видит, стоят ли слова запроса рядом. Координатный индекс из лекции 2 это умеет.
Попробуем простой вариант: переранжируем топ-100 BM25 и добавим бонус за пары терминов запроса, стоящие близко
(расстояние d ≤ 5 слов даёт вклад min(IDF) / d², идея из Rasolofo & Savoy, 2003).

In [22]:
positions = []  # для каждого документа: термин -> список позиций (координатный индекс «наизнанку»)
for toks in full_toks:
    pos = collections.defaultdict(list)
    for p, t in enumerate(toks):
        pos[t].append(p)
    positions.append(pos)

idf_of = {t: IDF[j] for t, j in vocab.items()}


def pair_proximity(i, q_terms, max_dist=5):
    bonus, terms = 0.0, sorted(q_terms)
    for x in range(len(terms)):
        for y in range(x + 1, len(terms)):
            pa, pb = positions[i].get(terms[x]), positions[i].get(terms[y])
            if not pa or not pb:
                continue
            for a in pa:
                for c in pb:
                    if abs(a - c) <= max_dist:
                        bonus += min(idf_of[terms[x]], idf_of[terms[y]]) / abs(a - c) ** 2
    return bonus


W_best = bm25_matrix(best_k1, best_b)


def proximity_run(qids, weight, k=100):
    run = {}
    for q in qids:
        cols_q = [vocab[t] for t in set(Q[q]) if t in vocab]
        s = np.asarray(W_best[:, cols_q].sum(axis=1)).ravel()
        cand = np.argpartition(-s, k)[:k]
        terms = {t for t in Q[q] if t in vocab}
        final = {i: s[i] + weight * pair_proximity(i, terms) for i in cand}
        run[q] = [doc_ids[i] for i in sorted(final, key=final.get, reverse=True)]
    return run


prox_grid = {w: evaluate(proximity_run(dev, w), dev)["nDCG@10"] for w in (0, 0.1, 0.25, 0.5, 1.0)}
print("nDCG@10 на dev по весу бонуса:", prox_grid)

nDCG@10 на dev по весу бонуса: {0: 0.665, 0.1: 0.658, 0.25: 0.662, 0.5: 0.645, 1.0: 0.626}


In [23]:
# почему так? посмотрим, какие документы бонус поднимает
q = next(q for q in dev if len(Q[q]) >= 4)
print(queries[q])
for d in proximity_run([q], 0.5)[q][:5]:
    i = doc_ids.index(d)
    print(f"  rel={qrels[q].get(d, '?')} бонус={pair_proximity(i, set(Q[q])):.2f}  {docs[d][0]}: {docs[d][1][:80]}")

Можно излечиться от наркотической зависимости полностью?
  rel=0 бонус=15.69  2ва Самолёта: В 1997 году Вадим Покровский покинул группу, пытаясь излечиться от наркотической
  rel=? бонус=4.93  Вавилон-5: Пагубные привычки играют важную роль в повествовании о «Вавилоне-5». Властолюбие
  rel=? бонус=0.00  Метамфетамин: Метамфетами́н (сокр. от N-метил-альфа-метилфенилэтиламин) — производное амфетами
  rel=0 бонус=4.34  Институт мозга человека имени Н. П. Бехтеревой РАН: В конце 1990-х годов в институте начались практики лечения героиновых и опиумных
  rel=1 бонус=5.09  Лечение наркомании: Метод был разработан в 80-х годах XX века как лечение алкоголизма, и 20 апреля 1


Бонус за близость здесь **не помогает**: любой ненулевой вес хуже, чем без бонуса, а при большом весе nDCG@10 на dev падает с 0.665 до 0.626.

Пример выше показывает почему. Больше всех бонус получил пассаж из биографии музыканта, где дословно стоит «излечиться от наркотической зависимости» — фраза совпала, но это не ответ на вопрос. Размеченные нерелевантные в MIRACL — это как раз такие «трудные» пассажи, найденные поиском: слова запроса в них стоят рядом. А пассажи короткие, и в релевантных слова запроса и так почти всегда близко.

В длинных документах близость различает больше. Проверяйте её на валидации ДЗ2, а не верьте на слово ни лекции, ни этому семинару.

## 7. whoosh: BM25F с весами полей

В части I мы видели, что whoosh по умолчанию ранжирует BM25F. Там же можно задать вес поля (boost) и свой `B` для каждого поля.
Проиндексируем уже лемматизированные поля: анализатор просто режет по пробелам, чтобы whoosh не лемматизировал по-своему.

In [24]:
import shutil

import whoosh.index as whoosh_index
from whoosh import scoring
from whoosh.analysis import RegexTokenizer
from whoosh.fields import ID, TEXT, Schema
from whoosh.query import Or, Term

analyzer = RegexTokenizer(r"\S+")
schema = Schema(docid=ID(stored=True), title=TEXT(analyzer=analyzer), body=TEXT(analyzer=analyzer))

INDEX_DIR = "indexdir_bm25"
shutil.rmtree(INDEX_DIR, ignore_errors=True)
os.mkdir(INDEX_DIR)
ix = whoosh_index.create_in(INDEX_DIR, schema)
start = time.time()
writer = ix.writer(limitmb=512)
for d, t, b in zip(doc_ids, title_toks, body_toks):
    writer.add_document(docid=d, title=" ".join(t), body=" ".join(b))
writer.commit()
print(f"проиндексировали за {time.time() - start:.0f} сек")

проиндексировали за 8 сек


In [25]:
def whoosh_run(qids, weighting, title_boost=1.0, k=100):
    run = {}
    with ix.searcher(weighting=weighting) as searcher:
        for q in qids:
            terms = set(Q[q])
            query = Or([Term("title", t, boost=title_boost) for t in terms] + [Term("body", t) for t in terms])
            run[q] = [hit["docid"] for hit in searcher.search(query, limit=k)]
    return run


start = time.time()
runs["whoosh BM25F по умолчанию"] = whoosh_run(test, scoring.BM25F())
runs[f"whoosh BM25F, title x{best_w}"] = whoosh_run(test, scoring.BM25F(), title_boost=best_w)
print(f"{time.time() - start:.0f} сек")
pd.DataFrame({name: evaluate(run, test) for name, run in runs.items() if name.startswith("whoosh")}).T

68 сек


,MAP,nDCG@10,MRR@10,Recall@100
whoosh BM25F по умолчанию,0.545,0.638,0.643,0.966
"whoosh BM25F, title x3",0.499,0.591,0.624,0.945


whoosh по умолчанию даёт 0.638 — меньше нашей склейки (0.661) и почти как смесь (b) (0.643). А с весом заголовка 3 проседает до 0.591.

Причина видна в исходниках `whoosh/scoring.py`: класс `BM25F` на самом деле считает **обычный BM25 по каждому полю отдельно** (со своим IDF и своей средней длиной поля), а запрос `Or` складывает скоры полей. Это вариант (b), а не BM25F из лекции. Вес поля умножает уже насыщенный скор, поэтому x3 для заголовка — это x3 к целому слагаемому, и пассажи одной статьи с общим заголовком вытесняют всё остальное.

Мораль: название функции — не гарантия формулы. Для ДЗ2 BM25F лучше написать самим — это десять строк, как в разделе 4.

## 8. Сводная таблица (test)

In [26]:
summary = pd.DataFrame({name: evaluate(run, test) for name, run in runs.items()}).T
summary.sort_values("nDCG@10", ascending=False)

,MAP,nDCG@10,MRR@10,Recall@100
BM25F (w_title=3),0.587,0.677,0.672,0.973
"BM25 (k1=1.2, b=0.75)",0.569,0.661,0.659,0.969
"BM25 (k1=0.9, b=0.5)",0.568,0.660,0.653,0.970
LM Елинек — Мерсер (λ=0.7),0.566,0.654,0.651,0.966
LM Дирихле (μ=200),0.558,0.649,0.641,0.968
смесь BM25(title)·1 + BM25(text),0.549,0.643,0.636,0.974
whoosh BM25F по умолчанию,0.545,0.638,0.643,0.966
TF-IDF (часть I),0.531,0.626,0.622,0.968
"whoosh BM25F, title x3",0.499,0.591,0.624,0.945


- TF-IDF → BM25: +3.5 пункта nDCG@10 только за счёт формулы.
- BM25F с правильными зонами даёт ещё +1.6 пункта. Наивная смесь BM25 по полям, в том числе whoosh-овский BM25F, наоборот, хуже склейки.
- Языковые модели почти догоняют BM25, но не обгоняют.
- Близость слов на коротких пассажах с «трудными» негативами не помогает.
- Recall@100 у всех около 0.97: кандидатов находят все, вся разница — в порядке внутри топа.

### Значима ли разница?

Метрика — среднее по запросам, и у каждого метода есть «удачные» и «неудачные» запросы. Разница в третьем знаке может оказаться шумом.
Проверяем **парным** тестом: сравниваем два метода на одних и тех же запросах.

- парный t-тест на позапросной разнице nDCG@10 (`scipy.stats.ttest_rel`);
- перестановочный (randomization) тест: если методы одинаковы, то в каждой паре можно случайно поменять их местами. Как часто случайная перестановка даёт разницу не меньше наблюдаемой — это и есть p-value.

Smucker, Allan, Carterette (CIKM 2007) показали, что на IR-метриках эти два теста дают почти одинаковые выводы.

In [27]:
from scipy import stats


def per_query_ndcg(run, qids, k=10):
    return np.array([ndcg([qrels[q].get(d, 0) for d in run[q]], list(qrels[q].values()), k) for q in qids])


def compare(name_a, name_b, n_perm=10_000):
    a, b = per_query_ndcg(runs[name_a], test), per_query_ndcg(runs[name_b], test)
    diff = a - b
    signs = np.random.default_rng(0).choice([-1, 1], size=(n_perm, len(diff)))  # случайно меняем методы местами
    p_perm = np.mean(np.abs((signs * diff).mean(axis=1)) >= abs(diff.mean()))
    fmt = lambda p: "< 0.0001" if p < 1e-4 else round(p, 4)
    return {"Δ nDCG@10": round(diff.mean(), 4), "p (t-тест)": fmt(stats.ttest_rel(a, b).pvalue),
            "p (перестановки)": fmt(p_perm), "запросов: A лучше / B лучше": f"{(diff > 0).sum()} / {(diff < 0).sum()}"}


pairs = [("BM25 (k1=1.2, b=0.75)", "TF-IDF (часть I)"),
         (f"BM25F (w_title={best_w})", "BM25 (k1=1.2, b=0.75)"),
         (f"LM Дирихле (μ={best_mu})", "BM25 (k1=1.2, b=0.75)")]
if (best_k1, best_b) != (1.2, 0.75):
    pairs.insert(1, (f"BM25 (k1={best_k1}, b={best_b})", "BM25 (k1=1.2, b=0.75)"))
pd.DataFrame({f"{a}  vs  {b}": compare(a, b) for a, b in pairs}).T

,Δ nDCG@10,p (t-тест),p (перестановки),запросов: A лучше / B лучше
"BM25 (k1=1.2, b=0.75) vs TF-IDF (часть I)",0.036,< 0.0001,< 0.0001,376 / 224
"BM25 (k1=0.9, b=0.5) vs BM25 (k1=1.2, b=0.75)",-0.0016,0.6643,0.666,233 / 250
"BM25F (w_title=3) vs BM25 (k1=1.2, b=0.75)",0.0157,0.0011,0.0014,285 / 244
"LM Дирихле (μ=200) vs BM25 (k1=1.2, b=0.75)",-0.0122,0.0151,0.0154,250 / 315


- BM25 лучше TF-IDF однозначно: p < 0.0001.
- Подобранные на dev k1 = 0.9, b = 0.5 от дефолтных **не отличаются** (p ≈ 0.67): на 233 запросах лучше, на 250 хуже. Это шум, а не улучшение.
- BM25F лучше склейки значимо (p ≈ 0.001), хотя выигрыш всего 1.6 пункта: 852 запросов хватает, чтобы его заметить.
- LM Дирихле хуже BM25 тоже значимо (p ≈ 0.015), при разнице в 1.2 пункта.

t-тест и перестановочный тест дают почти одинаковые p. Две оговорки. На 50 запросах такие же разницы утонули бы в шуме.
А если сравнить с базой 20 бесполезных вариантов, один из них в среднем окажется «значимым» при p < 0.05 — это проблема множественных сравнений.
Для ДЗ2: если выигрыш на вашей валидации в третьем знаке, проверьте его этим тестом, прежде чем верить.

## 9. Задания

**Задание 1. BM25+.** Lv & Zhai (2011) заметили, что у BM25 очень длинный документ с одним вхождением термина получает почти нулевой вклад —
меньше, чем документ без термина, но с лучшими остальными словами. BM25+ добавляет к насыщенной части константу δ (обычно 1.0) для каждого совпавшего термина:
IDF · (tf·(k1+1) / (tf + k1·(...)) + δ). Реализуйте `bm25plus_matrix`, подберите δ на dev и сравните с BM25 на test.

**Задание 2. Гибрид BM25F + LM через RRF.** Возьмите выдачи BM25F и LM Дирихле и сфьюзьте их по Reciprocal Rank Fusion:
RRF(d) = Σ 1 / (60 + rank(d)). Стало ли лучше? Сравните с выводом из части I про RRF двух TF-IDF.

**Задание 3. Где LM лучше BM25.** Найдите запросы, где nDCG@10 у LM Дирихле заметно выше, чем у BM25, и наоборот.
Посмотрите на длины запросов и документов в топе. Сформулируйте гипотезу.

In [27]:
# Задание 1: ваш код

In [28]:
# Задание 2: ваш код

In [29]:
# Задание 3: ваш код

## 10. Что дальше: ДЗ2

Контест на Kaggle по датасету **VK MARCO**: реальные запросы и веб-документы с зонами TITLE и BODY. Надо переранжировать документы, используя только их тексты.

- Бейзлайн: TF-IDF по склейке title + body.
- **Можно:** всё из двух лекций — TF-IDF, BM25, BM25F, языковые модели, близость слов, лемматизация, подбор коэффициентов на трейне.
- **Нельзя:** ML (кроме подбора коэффициентов), готовые библиотеки с ранжирующей формулой (`rank-bm25`, `bm25s`, whoosh-скоринг и т.п.), тем более нейросети.
- Решение — `solution.py`, который воспроизводит сабмишн, плюс `requirements.txt`.

Сегодняшний ноутбук — хорошая заготовка: матричный BM25/BM25F пишется сам, а `bm25s` пригодится только чтобы сверить, что формула без ошибок.
Документы в VK MARCO длиннее пассажей Википедии, поэтому нормировка длины и близость слов там могут вести себя иначе, чем здесь, — проверяйте на своей валидации.

Следующая лекция — поведенческое и ссылочное ранжирование.

In [28]:
shutil.rmtree(INDEX_DIR, ignore_errors=True)  # убираем индекс whoosh